# 00 — Setup & Configuration

Mounts ADLS Gen2, creates Bronze/Silver/Gold databases.

**Run this first before any other notebook.**

In [ ]:
# ============================================================
# FILL IN YOUR VALUES BELOW
# ============================================================
STORAGE_ACCOUNT = "stgamingdatalake"      # your storage account name
CLIENT_ID       = "<your-app-client-id>"   # Service Principal App ID
CLIENT_SECRET   = "<your-client-secret>"   # Service Principal Secret
TENANT_ID       = "<your-tenant-id>"       # Azure Tenant ID
CONTAINER       = "gaming-data"            # container name in ADLS
# ============================================================
print("Config loaded")

In [ ]:
# Mount ADLS Gen2 using OAuth (Service Principal)
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id":
        CLIENT_ID,
    "fs.azure.account.oauth2.client.secret":
        CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token"
}

# Unmount if already mounted (safe to re-run)
try:
    dbutils.fs.unmount("/mnt/gaming")
    print("Previous mount removed")
except Exception:
    pass

dbutils.fs.mount(
    source=f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/",
    mount_point="/mnt/gaming",
    extra_configs=configs
)
print("Mount successful ✅")

In [ ]:
# Create the three Medallion databases
spark.sql("CREATE DATABASE IF NOT EXISTS gaming_bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS gaming_silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gaming_gold")
print("Databases created ✅")
spark.sql("SHOW DATABASES").show()

In [ ]:
# Verify the raw CSV is visible
display(dbutils.fs.ls("/mnt/gaming/raw/"))